# 05 — Fraud & Risk Detection

- **Isolation Forest** anomaly detection on behavioral features
- Post-hoc evaluation against known fraud cases
- **Composite 0-100 risk score** combining delinquency, NSF, utilization, anomaly flag
- Risk score documentation for regulatory compliance

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_pipeline
from src import features as feat
from src.models import risk_detector as rd

pd.set_option('display.max_columns', 50)

In [ ]:
con, counts = load_pipeline(verbose=True)

## 1. Build Feature Tables

In [ ]:
customer_base = feat.build_customer_base(con)
anomaly_features = feat.build_anomaly_features(con)

print(f"customer_base: {customer_base.shape}")
print(f"anomaly_features: {anomaly_features.shape}")
print(f"\nFraud cases in customer_base: {customer_base['has_fraud_case'].sum()}")

## 2. Train Isolation Forest

**Key design decision**: `has_fraud_case` is EXCLUDED from training features.
It's used only for post-hoc evaluation. Including it would turn unsupervised
anomaly detection into supervised classification, which defeats the purpose.

In [ ]:
print("Training Isolation Forest...")
iso_model, iso_scaler, anomaly_preds = rd.train_isolation_forest(
    anomaly_features,
    fraud_rate_df=customer_base,
    save_path='../outputs/saved_models/isolation_forest.joblib',
    scaler_path='../outputs/saved_models/isolation_forest_scaler.joblib'
)

In [ ]:
# Anomaly distribution
print(f"\nAnomaly flag distribution:")
print(anomaly_preds['anomaly_flag'].value_counts())

## 3. Evaluate Against Known Fraud Cases

In [ ]:
eval_metrics = rd.evaluate_anomaly_detector(anomaly_preds, customer_base)

In [ ]:
# Anomaly scatter: Utilization vs Avg Transaction Amount
# Join utilization from customer_base
scatter_data = anomaly_preds.merge(
    customer_base[['current_account_nbr', 'ca_current_utilz', 'has_fraud_case']],
    on='current_account_nbr', how='left'
)
scatter_data['ca_current_utilz'] = pd.to_numeric(scatter_data['ca_current_utilz'], errors='coerce')

fig = px.scatter(
    scatter_data,
    x='ca_current_utilz', y='avg_txn_amt',
    color=scatter_data['anomaly_flag'].map({0: 'Normal', 1: 'Anomaly'}),
    color_discrete_map={'Normal': '#3498db', 'Anomaly': '#e74c3c'},
    title='Anomaly Detection — Utilization vs Avg Transaction Amount',
    labels={'ca_current_utilz': 'Utilization (%)', 'avg_txn_amt': 'Avg Transaction Amount ($)'},
    opacity=0.6
)

# Overlay known fraud cases
fraud_data = scatter_data[scatter_data['has_fraud_case'] == 1]
if len(fraud_data) > 0:
    fig.add_trace(go.Scatter(
        x=fraud_data['ca_current_utilz'], y=fraud_data['avg_txn_amt'],
        mode='markers', name='Known Fraud',
        marker=dict(size=12, symbol='x', color='#f39c12', line=dict(width=2))
    ))

fig.update_layout(height=600, width=800, template='plotly_white')
fig.write_image("../outputs/figures/anomaly_scatter.png", scale=2)
fig.show()

## 4. Compute Composite Risk Score

**Components (interpretable, documented weights):**

| Component | Weight | Source |
|-----------|--------|--------|
| Delinquency depth | 30% | `cu_nbr_days_dlq` normalized 0-100 |
| Utilization level | 25% | `ca_current_utilz` |
| Anomaly flag | 20% | Isolation Forest binary |
| NSF frequency | 15% | `ca_nsf_count_lst_12_months` |
| Payment history | 10% | Max delinquency level from parsed string |

In [ ]:
print("\nComputing composite risk score...")
customer_with_risk = rd.compute_risk_score(customer_base, anomaly_preds)

In [ ]:
# Risk score distribution
fig = px.histogram(
    customer_with_risk, x='risk_score', nbins=50,
    title='Composite Risk Score Distribution (0-100)',
    labels={'risk_score': 'Risk Score'},
    color_discrete_sequence=['#e74c3c']
)
fig.update_layout(height=400, width=700, template='plotly_white')
fig.write_image("../outputs/figures/risk_score_distribution.png", scale=2)
fig.show()

## 5. Risk Score by Segment

In [ ]:
# Load segmented data
from src.models import segmentation as seg_module
customer_segmented = seg_module.assign_rule_based_segments(customer_with_risk)

fig = px.box(
    customer_segmented, x='segment_name', y='risk_score',
    color='segment_name',
    color_discrete_map=seg_module.SEGMENT_COLORS,
    title='Risk Score Distribution by Segment',
    labels={'segment_name': 'Segment', 'risk_score': 'Risk Score'}
)
fig.update_layout(height=500, width=800, template='plotly_white', showlegend=False)
fig.write_image("../outputs/figures/risk_by_segment.png", scale=2)
fig.show()

In [ ]:
# Risk score stats by segment
risk_by_seg = customer_segmented.groupby('segment_name')['risk_score'].describe().round(2)
print("\nRisk Score by Segment:")
print(risk_by_seg.to_string())

## 6. Save Risk Data

In [ ]:
customer_with_risk.to_parquet('../outputs/predictions/customer_with_risk.parquet', index=False)
anomaly_preds.to_parquet('../outputs/predictions/anomaly_predictions.parquet', index=False)

print("Saved:")
print("  - customer_with_risk.parquet")
print("  - anomaly_predictions.parquet")